# Generate All Paper Figures

In [ ]:
# =============================================================================
# Master figure-generation notebook.
# Loads all pre-computed pkl files and produces every paper figure + Table 2.
# Run this after all upstream notebooks (03, 04a, 04b, 05a) have completed.
# =============================================================================

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import pickle
import numpy as np
import matplotlib.pyplot as plt
import os

from pdspl_utils.plotting import plot_dspl_corner
from pdspl_utils.pairing import (
    plot_dataset_corner,
    plot_reldiff_corner,
    plot_beta_E_vs_D_MC,
    plot_pairing_scatter,
    generate_latex_summary_table,
)

# %load_ext autoreload
# %autoreload 2

FIGURE_DIRECTORY = "../figures/all_paper_figures"
os.makedirs(FIGURE_DIRECTORY, exist_ok=True)

In [ ]:
# ── Cell 2: Load all pre-computed data ───────────────────────────────────────
with open("../data/samples/pdspl_samples_with_pairs.pkl", "rb") as f:
    pdspl_samples = pickle.load(f)

with open("../data/samples/mc_results_pairing.pkl", "rb") as f:
    mc_results = pickle.load(f)

with open("../data/samples/forecast_scenarios_w0waCDM_fixed_scatter.pkl", "rb") as f:
    fixed_scenarios = pickle.load(f)

with open("../data/samples/forecast_scenarios_w0waCDM_free_scatter_lsst_y10.pkl", "rb") as f:
    free_scenarios = pickle.load(f)

# Experiment 05a data
with open("../data/samples/mc_results_photo_vs_specz.pkl", "rb") as f:
    mc_results_expt = pickle.load(f)

with open("../data/samples/forecast_scenarios_photo_vs_specz.pkl", "rb") as f:
    photo_specz_scenarios = pickle.load(f)

# pdspl_samples_expt for Figure 9 (must have 'pairs_analysis' key)
with open("../data/samples/pdspl_samples_expt_photo_vs_specz.pkl", "rb") as f:
    pdspl_samples_expt = pickle.load(f)

In [ ]:
# ── Cell 3: Accessible colour palette (Okabe-Ito) ────────────────────────────
C_SKYBLUE   = "#56B4E9"
C_ORANGE    = "#E69F00"
C_GREEN     = "#009E73"
C_VERMILION = "#D55E00"
C_PURPLE    = "#CC79A7"
C_BLUE      = "#0072B2"
C_BLACK     = "#000000"

accessible_colors = {
    "lsst_y1":                   C_ORANGE,
    "lsst_y10":                  C_GREEN,
    "lsst_y10_baseline":         C_GREEN,
    "lsst_4most_spec-z":         C_PURPLE,
    "lsst_4most_spec-z_sigma_v": C_BLUE,
    "DSPL":                      C_SKYBLUE,
    "lsst_y10_om_prior":         C_BLACK,
    "DSPL_om_prior":             C_BLUE,
    "lsst_y10_photo_z":          C_GREEN,
    "lsst_y10_spec_z":           C_BLUE,
    "lsst_y10_zD_spec":          C_SKYBLUE,
    "lsst_y10_all_spec":         C_PURPLE,
}
accessible_markers = {
    "lsst_y1":                   "^",
    "lsst_y10":                  "o",
    "lsst_4most_spec-z":         "s",
    "lsst_4most_spec-z_sigma_v": "D",
    "lsst_y10_photo_z":          "o",
    "lsst_y10_spec_z":           "s",
}

truth = {
    "h0": 70.0, "om": 0.3, "w0": -1.0, "wa": 0.0,
    "lambda_int":  1.0,  "lambda_sigma": 0.05,
    "gamma_pl":    2.0,  "gamma_sigma":  0.16,
}
fixed_params = {"h0": 70.0}
latex_labels = {
    "om":           r"$\Omega_{\rm m}$",
    "w0":           r"$w_0$",
    "wa":           r"$w_a$",
    "lambda_int":   r"$\overline{\lambda}_{\rm MST}$",
    "lambda_sigma": r"$\sigma({\lambda}_{\rm MST})$",
    "gamma_pl":     r"$\overline{\gamma}_{\rm pl}$",
    "gamma_sigma":  r"$\sigma({\gamma}_{\rm pl})$",
    "beta_c0":      r"$\sigma_{\beta_{\rm E},\rm \mathcal{D}}^{(0)}$",
    "beta_c1":      r"$\sigma_{\beta_{\rm E},\rm \mathcal{D}}^{(1)}$",
}
custom_ranges = [
    (0, 1), (-2, 0), (-3, 3),
    (0.8, 1.2), (0.0, 0.18),
    (1.8, 2.2), (0.0, 0.32),
]

In [ ]:
# ── Cell 4: Figure 3 — Unpaired population corner plot ───────────────────────
print("Generating Figure 3: unpaired GGL population corner plot...")
fig3 = plot_dataset_corner(
    pdspl_samples,
    samples_to_plot=["lsst_y1", "lsst_y10", "lsst_4most_spec-z", "lsst_4most_spec-z_sigma_v"],
    key_list=["z_D", "z_S", "theta_E", "sigma_v_D", "e_mass_D", "mag_D_i", "mag_S_i_lensed"],
    key_latex_labels={
        "z_D":            r"$z_D$",
        "z_S":            r"$z_S$",
        "theta_E":        r"$\theta_E$",
        "sigma_v_D":      r"$\sigma_{v, D}$",
        "e_mass_D":       r"$q_{mass, D}$",
        "mag_D_i":        r"$m_{D, i}$",
        "mag_S_i_lensed": r"$m_{S, i}^{lensed}$",
    },
    plot_ranges=[(0.0, 2.5), (0.0, 5.0), (0.0, 2.5), (150, 450), (0, 0.4), (17, 27), (18, 26)],
    save_path=f"{FIGURE_DIRECTORY}/slsim_corner_GGL_all_samples_v2.pdf",
    custom_colors_dict=accessible_colors,
)
plt.close(fig3)
print("  → saved")

In [ ]:
# ── Cell 5: Figure 4 — β_E vs D (main samples, quadrature fit) ───────────────
print("Generating Figure 4: beta_E vs D (main samples)...")
fig4 = plot_beta_E_vs_D_MC(
    pdspl_samples, mc_results,
    fit_type="quadrature",
    custom_colors_dict=accessible_colors,
    custom_markers_dict=accessible_markers,
    save_path=f"{FIGURE_DIRECTORY}/beta_E_vs_D_MC.png",
)
plt.close(fig4)
print("  → saved")

In [ ]:




# ── Cell 5b: Figure 4b — Pairing scatter (lens 1 vs lens 2) ──────────────────
# Primary pairing-quality figure: lens 1 vs lens 2 scatter with y=x diagonal,
# plus KDE of relative differences, for all six pairing parameters.
print("Generating Figure 4b: pairing scatter (lens 1 vs lens 2)...")

_pairing_scatter_keys = [
    "z_D", "R_e_arcsec",
    "flux_D_i", "flux_ratio_D_gr",
    "flux_ratio_D_ri", "sigma_v_D",
]
_pairing_scatter_labels = {
    "z_D":             r"$z_D$",
    "R_e_arcsec":      r"$R_e$ [arcsec]",
    "flux_D_i":        r"$f_{D,i}$",
    "flux_ratio_D_gr": r"$f_{D,g}/f_{D,r}$",
    "flux_ratio_D_ri": r"$f_{D,r}/f_{D,i}$",
    "sigma_v_D":       r"$\sigma_{v,D}$ [km/s]",
}
_pairing_hist_labels = {
    "z_D":             r"$\Delta z_D / \langle z_D \rangle$",
    "R_e_arcsec":      r"$\Delta R_e / \langle R_e \rangle$",
    "flux_D_i":        r"$\Delta f_{D,i} / \langle f_{D,i} \rangle$",
    "flux_ratio_D_gr": r"$\Delta (f_{D,g}/f_{D,r}) / \langle f_{D,g}/f_{D,r} \rangle$",
    "flux_ratio_D_ri": r"$\Delta (f_{D,r}/f_{D,i}) / \langle f_{D,r}/f_{D,i} \rangle$",
    "sigma_v_D":       r"$\Delta \sigma_{v,D} / \langle \sigma_{v,D} \rangle$",
}

fig4b = plot_pairing_scatter(
    pdspl_samples=pdspl_samples,
    samples_to_plot=["lsst_y1", "lsst_y10", "lsst_4most_spec-z", "lsst_4most_spec-z_sigma_v"],
    pairing_param_keys=_pairing_scatter_keys,
    pairing_param_labels=_pairing_scatter_labels,
    pairing_hist_labels=_pairing_hist_labels,
    custom_colors_dict=accessible_colors,
    custom_markers_dict=accessible_markers,
    n_scatter_points=2000,
    save_path=f"{FIGURE_DIRECTORY}/pairing_scatter_lens1_vs_lens2.pdf",
    reldiff_range=(-0.25, 0.25),
)
fig4b.tight_layout()
plt.close(fig4b)
print("  → saved")

In [ ]:
# ── Cell 6: Table 2 — LaTeX pairing statistics ────────────────────────────────
print("\nGenerating Table 2 (LaTeX):")
generate_latex_summary_table(
    pdspl_samples,
    ["lsst_y1", "lsst_y10", "lsst_4most_spec-z", "lsst_4most_spec-z_sigma_v"],
)

In [ ]:
# ── Cell 7: Figure 5 — Relative-difference corner ────────────────────────────
print("\nGenerating Figure 5: relative-difference corner...")
fig5 = plot_reldiff_corner(
    pdspl_samples,
    samples_to_plot=["lsst_y1", "lsst_y10", "lsst_4most_spec-z", "lsst_4most_spec-z_sigma_v"],
    key_list=[
        "rel_diff_z_D", "rel_diff_sigma_v_D", "rel_diff_beta_E",
        "rel_diff_gamma_pl", "rel_diff_mag_D_i", "rel_diff_color_D_gr",
    ],
    key_latex_labels={
        "rel_diff_z_D":        r"$\Delta z_D / z_D$",
        "rel_diff_sigma_v_D":  r"$\Delta \sigma_{v, D} / \sigma_{v, D}$",
        "rel_diff_beta_E":     r"$\Delta \beta_E / \beta_E$",
        "rel_diff_gamma_pl":   r"$\Delta \gamma_{\rm pl} / \gamma_{\rm pl}$",
        "rel_diff_mag_D_i":    r"$\Delta m_{D, i} / m_{D, i}$",
        "rel_diff_color_D_gr": r"$\Delta c_{D, g-r} / c_{D, g-r}$",
    },
    custom_ranges={
        "rel_diff_z_D":        (-0.2,  0.2),
        "rel_diff_sigma_v_D":  (-0.5,  0.5),
        "rel_diff_beta_E":     (-0.5,  0.5),
        "rel_diff_gamma_pl":   (-0.4,  0.4),
        "rel_diff_mag_D_i":    (-0.3,  0.3),
        "rel_diff_color_D_gr": (-0.3,  0.3),
    },
    custom_colors_dict=accessible_colors,
    save_path=f"{FIGURE_DIRECTORY}/pairing_reldiff_corner_all_samples.pdf",
)
plt.close(fig5)
print("  → saved")

In [ ]:
# ── Cell 8: Figure 6 — LSST Y1 vs Y10 ───────────────────────────────────────
print("Generating Figure 6: LSST Y1 vs Y10 forecast...")
fig6 = plot_dspl_corner(
    fixed_scenarios, truth, fixed_params,
    scenarios_to_plot=["lsst_y1", "lsst_y10"],
    custom_ranges=custom_ranges, latex_labels=latex_labels, figsize=(14, 14),
    show_multiple_titles=True,
    custom_colors_dict=accessible_colors,
    custom_linestyles_dict={"lsst_y1": "--", "lsst_y10": "-"},
    save_path=f"{FIGURE_DIRECTORY}/pdspls_lsst_y1_y10_forecast_w0waCDM_fixed_scatter.pdf",
)
plt.close(fig6)
print("  → saved")

In [ ]:
# ── Cell 9: Figure 7 — LSST Y10 vs 4MOST ────────────────────────────────────
print("Generating Figure 7: LSST Y10 vs 4MOST forecast...")
fig7 = plot_dspl_corner(
    fixed_scenarios, truth, fixed_params,
    scenarios_to_plot=["lsst_4most_spec-z_sigma_v", "lsst_4most_spec-z", "lsst_y10"],
    custom_ranges=custom_ranges, latex_labels=latex_labels, figsize=(14, 14),
    show_multiple_titles=True,
    custom_colors_dict=accessible_colors,
    custom_linestyles_dict={
        "lsst_4most_spec-z_sigma_v": ":",
        "lsst_4most_spec-z":         "-.",
        "lsst_y10":                  "-",
    },
    save_path=f"{FIGURE_DIRECTORY}/pdspls_lsst_y10_vs_4MOST_forecast_w0waCDM_fixed_scatter.pdf",
)
plt.close(fig7)
print("  → saved")

In [ ]:
# ── Cell 10: Figure 8 — DSPL vs PDSPL (LSST Y10) with/without Ωm prior ──────
print("Generating Figure 8: DSPL vs PDSPL Y10 forecast...")
fig8 = plot_dspl_corner(
    fixed_scenarios, truth, fixed_params,
    scenarios_to_plot=["DSPL", "lsst_y10", "DSPL_om_prior", "lsst_y10_om_prior"],
    custom_ranges=custom_ranges, latex_labels=latex_labels, figsize=(14, 14),
    show_multiple_titles=True,
    custom_colors_dict=accessible_colors,
    custom_linestyles_dict={
        "DSPL":               "--",
        "lsst_y10":           "-",
        "DSPL_om_prior":      "--",
        "lsst_y10_om_prior":  "-",
    },
    save_path=f"{FIGURE_DIRECTORY}/pdspl_vs_dspl_forecast_w0waCDM_fixed_scatter.pdf",
)
plt.close(fig8)
print("  → saved")

In [ ]:
# ── Cell 11: Figure 9 — β_E vs D (photo-z vs spec-z, Appendix B) ─────────────
#
# Important: pass pdspl_samples_expt (with keys lsst_y10_photo_z / lsst_y10_spec_z)
# and the corresponding mc_results_expt — NOT the main pdspl_samples dict.
print("Generating Figure 9: beta_E vs D (photo-z vs spec-z)...")
fig9 = plot_beta_E_vs_D_MC(
    pdspl_samples_expt, mc_results_expt,
    fit_type="quadrature",
    custom_colors_dict=accessible_colors,
    custom_markers_dict=accessible_markers,
    save_path=f"{FIGURE_DIRECTORY}/beta_E_vs_D_LSST_Y10_photo_vs_spec-z.png",
)
plt.close(fig9)
print("  → saved")

In [ ]:
# ── Cell 12: Figure 10 — Photo-z vs spec-z cosmological forecast ──────────────
print("Generating Figure 10: photo-z vs spec-z cosmological forecast...")
fig10 = plot_dspl_corner(
    photo_specz_scenarios, truth, fixed_params,
    scenarios_to_plot=["lsst_y10_baseline", "lsst_y10_zD_spec", "lsst_y10_all_spec"],
    custom_ranges=custom_ranges, latex_labels=latex_labels, figsize=(14, 14),
    show_multiple_titles=True,
    custom_colors_dict=accessible_colors,
    custom_linestyles_dict={
        "lsst_y10_baseline": "-",
        "lsst_y10_zD_spec":  "--",
        "lsst_y10_all_spec": "-.",
    },
    save_path=f"{FIGURE_DIRECTORY}/lsst_y10_photoz_vs_specz_forecast.pdf",
)
plt.close(fig10)
print("  → saved")

In [ ]:
# ── Cell 13: Figures 11 & 12 — Free vs fixed scatter ─────────────────────────
print("Generating Figures 11 & 12: free scatter forecasts...")

# Retrieve calibrated quadrature fit coefficients to set truth lines
coeffs_y10 = pdspl_samples["lsst_y10"]["pairs_analysis"]["scatter_vs_dissimilarity_fit_coeffs"]
custom_truth_free = truth.copy()
custom_truth_free["beta_c0"] = coeffs_y10[1]   # c0_floor
custom_truth_free["beta_c1"] = coeffs_y10[0]   # c1_slope

custom_ranges_free = custom_ranges + [(0, 0.25), (0, 2.0)]  # beta_c0, beta_c1

# Figure 11: free scatter posteriors (w/ and w/o Ωm prior)
fig11 = plot_dspl_corner(
    free_scenarios, custom_truth_free, fixed_params,
    scenarios_to_plot=["lsst_y10", "lsst_y10_om_prior"],
    custom_ranges=custom_ranges_free,
    latex_labels=latex_labels, figsize=(14, 14),
    show_multiple_titles=True,
    save_path=f"{FIGURE_DIRECTORY}/free_scatter_lsst_y10_forecast_w0waCDM.pdf",
    titles_fontsize=12,
)
plt.close(fig11)
print("  → Figure 11 saved")

# Figure 12: fixed vs free scatter comparison
merged = {}
for k in ["lsst_y10", "lsst_y10_om_prior"]:
    merged[f"fixed_{k}"] = {
        "name":    fixed_scenarios[k]["name"] + " (Fixed Scat.)",
        "samples": fixed_scenarios[k]["samples"][:, :7],
        "color":   C_GREEN if "om" not in k else C_BLACK,
    }
    merged[f"free_{k}"] = {
        "name":    free_scenarios[k]["name"] + " (Free Scat.)",
        "samples": free_scenarios[k]["samples"][:, :7],
        "color":   C_ORANGE if "om" not in k else C_VERMILION,
    }

fig12 = plot_dspl_corner(
    merged, truth, fixed_params,
    scenarios_to_plot=[
        "fixed_lsst_y10", "free_lsst_y10",
        "fixed_lsst_y10_om_prior", "free_lsst_y10_om_prior",
    ],
    custom_ranges=custom_ranges, latex_labels=latex_labels, figsize=(14, 14),
    show_multiple_titles=True,
    custom_linestyles_dict={
        "fixed_lsst_y10":           "-",
        "free_lsst_y10":            "--",
        "fixed_lsst_y10_om_prior":  "-",
        "free_lsst_y10_om_prior":   "--",
    },
    save_path=f"{FIGURE_DIRECTORY}/free_vs_fixed_scatter_lsst_y10_forecast_w0waCDM.pdf",
)
plt.close(fig12)
print("  → Figure 12 saved")

print(f"\n✓ All figures saved to: {FIGURE_DIRECTORY}")